# Term-Deposit Campaign Targeting: Predictive Analytics Case Study

**Portfolio notebook**  
Prepared by **Ashutosh Girish Shinde**  
MSc Business Analytics

This notebook demonstrates an end-to-end business analytics workflow for prioritising prospects in a telemarketing campaign for a fixed-term deposit product.

> **Portfolio data note:** the raw dataset is not distributed in the GitHub repository. To run this notebook, place the dataset at `data/term_deposit_campaign_data.csv` with the same schema used below.

## 1. Business objective

The commercial objective is to reduce unproductive outbound calls while continuing to identify customers who are more likely to subscribe. The model is therefore treated as a **prospect-ranking tool** rather than a fully automated rejection system.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
import joblib

RANDOM_STATE = 0
DATA_PATH = "data/term_deposit_campaign_data.csv"

## 2. Load and validate the dataset

The dataset contains historical call records. Each record is treated independently because no unique customer identifier is available.

In [ ]:
df = pd.read_csv(DATA_PATH)

expected_columns = [
    "age", "job", "marital", "education", "default", "balance", "housing", "loan",
    "contact", "day", "duration", "campaign", "pdays", "previous", "poutcome", "y"
]
missing = set(expected_columns) - set(df.columns)
unexpected = set(df.columns) - set(expected_columns)

print(f"Dataset shape: {df.shape}")
print(f"Missing expected columns: {sorted(missing)}")
print(f"Unexpected columns: {sorted(unexpected)}")
print()
print("Target distribution:")
print(df["y"].value_counts().to_frame("count"))
print()
print("Subscription rate:")
print(round((df["y"] == "yes").mean(), 3))

## 3. Leakage control

`duration` is excluded from predictive modelling because it is only known after a call has occurred. It can be useful for descriptive analysis, but including it in a pre-call prospecting model would introduce target leakage.

In [ ]:
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

X = df.drop(columns=[TARGET] + LEAKAGE_COLUMNS)
y = df[TARGET]

categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

## 4. Descriptive analysis

The descriptive section focuses on conversion rates by commercially interpretable groups

In [ ]:
def conversion_table(data, group_col, target="y"):
    table = (
        data.groupby(group_col)[target]
        .agg(total="count", subscribers=lambda s: (s == "yes").sum())
        .assign(conversion_rate=lambda x: x["subscribers"] / x["total"])
        .sort_values("conversion_rate", ascending=False)
    )
    return table

for col in ["poutcome", "contact", "housing", "loan", "education", "job"]:
    print()
    print(f"Conversion by {col}")
    display(conversion_table(df, col).round(3))

In [ ]:
# Create age, balance and campaign-contact bands for interpretable descriptive analysis.
edf = df.copy()
edf["age_group"] = pd.cut(
    edf["age"],
    bins=[18, 25, 35, 45, 55, 65, 100],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    right=False,
)
edf["balance_band"] = pd.cut(
    edf["balance"],
    bins=[-np.inf, 0, 500, 1000, 2000, 5000, np.inf],
    labels=["Negative", "0-500", "500-1k", "1k-2k", "2k-5k", "5k+"],
)
edf["campaign_band"] = pd.cut(
    edf["campaign"],
    bins=[0, 1, 2, 3, 5, 10, np.inf],
    labels=["1", "2", "3", "4-5", "6-10", "11+"],
)

summary_groups = ["age_group", "balance_band", "campaign_band"]
for col in summary_groups:
    print()
    print(f"Conversion by {col}")
    display(conversion_table(edf, col).round(3))

In [ ]:
# Visual summary of selected conversion rates.
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
selected = ["poutcome", "contact", "housing", "campaign_band"]

for ax, col in zip(axes.ravel(), selected):
    tab = conversion_table(edf, col).sort_values("conversion_rate", ascending=True)
    ax.barh(tab.index.astype(str), tab["conversion_rate"])
    ax.set_title(f"Conversion rate by {col}")
    ax.set_xlabel("Subscription rate")
    ax.set_xlim(0, max(0.9, tab["conversion_rate"].max() + 0.05))

plt.tight_layout()
plt.show()

## 5. Train/holdout split and preprocessing

A stratified 70/30 split is used to preserve class balance in both development and holdout samples. Model selection is performed only within the development sample.

In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Development sample:", X_train.shape, y_train.value_counts(normalize=True).round(3).to_dict())
print("Holdout sample:", X_holdout.shape, y_holdout.value_counts(normalize=True).round(3).to_dict())

preprocess_tree = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

preprocess_lr = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features),
    ]
)

## 6. Model comparison

The models selected represent different operating policies:

- **Majority benchmark:** checks whether the model adds value beyond always predicting `no`.
- **Decision Tree:** interpretable first-cut segmentation model.
- **Logistic Regression:** linear probability model with class balancing.
- **Random Forest:** non-linear ensemble model for stronger ranking performance.

In [ ]:
models = {
    "Majority benchmark": Pipeline([
        ("preprocess", preprocess_tree),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "Decision Tree": Pipeline([
        ("preprocess", preprocess_tree),
        ("model", DecisionTreeClassifier(
            criterion="entropy",
            max_depth=4,
            min_samples_leaf=7,
            random_state=RANDOM_STATE,
        )),
    ]),
    "Logistic Regression": Pipeline([
        ("preprocess", preprocess_lr),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocess_tree),
        ("model", RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
}

In [ ]:
# Use a binary target for cross-validation scoring so that the positive class is explicit.
y_train_binary = (y_train == "yes").astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train_binary,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
            "roc_auc": "roc_auc",
            "pr_auc": "average_precision",
        },
        n_jobs=-1,
        error_score="raise",
    )
    cv_rows.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
        "roc_auc": scores["test_roc_auc"].mean(),
        "pr_auc": scores["test_pr_auc"].mean(),
    })

cv_results = pd.DataFrame(cv_rows).set_index("model").round(3)
display(cv_results)

In [ ]:
ax = cv_results[["precision", "recall", "f1", "roc_auc", "pr_auc"]].plot(
    kind="bar",
    figsize=(12, 5),
    rot=25,
)
ax.set_title("Five-fold cross-validation performance on development sample")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Holdout evaluation of selected model

Random Forest is selected because it provides the strongest balance between precision, recall and ranking metrics. The holdout sample is evaluated once after model selection.

In [ ]:
selected_model = models["Random Forest"]
selected_model.fit(X_train, y_train)

y_pred = selected_model.predict(X_holdout)
y_proba = selected_model.predict_proba(X_holdout)[:, list(selected_model.classes_).index("yes")]
y_holdout_binary = (y_holdout == "yes").astype(int)

holdout_metrics = pd.DataFrame({
    "metric": ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"],
    "value": [
        accuracy_score(y_holdout, y_pred),
        precision_score(y_holdout, y_pred, pos_label="yes", zero_division=0),
        recall_score(y_holdout, y_pred, pos_label="yes"),
        f1_score(y_holdout, y_pred, pos_label="yes"),
        roc_auc_score(y_holdout_binary, y_proba),
        average_precision_score(y_holdout_binary, y_proba),
    ],
}).round(3)

display(holdout_metrics)
print(classification_report(y_holdout, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_holdout, y_pred, labels=["no", "yes"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["no", "yes"])
disp.plot(values_format="d")
plt.title("Random Forest holdout confusion matrix")
plt.tight_layout()
plt.show()

## 8. Model interpretation

Permutation importance is calculated on the holdout sample using F1 as the performance measure. This estimates which features the fitted model relies on most strongly in this dataset.

In [ ]:
perm = permutation_importance(
    selected_model,
    X_holdout,
    y_holdout,
    scoring="f1",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": X_holdout.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance_df.round(4))

ax = importance_df.head(10).sort_values("importance_mean").plot(
    kind="barh",
    x="feature",
    y="importance_mean",
    figsize=(9, 5),
    legend=False,
)
ax.set_title("Permutation importance on holdout sample")
ax.set_xlabel("Mean decrease in F1")
plt.tight_layout()
plt.show()

## 9. Probability ranking for campaign targeting

A ranking approach is more useful than a rigid classification threshold because the call centre can match the number of calls to available capacity.

In [ ]:
ranked = pd.DataFrame({
    "actual_yes": y_holdout_binary.values,
    "p_yes": y_proba,
}).sort_values("p_yes", ascending=False).reset_index(drop=True)

rows = []
for share in [0.10, 0.20, 0.30, 0.40, 0.50]:
    n = int(round(len(ranked) * share))
    selected = ranked.iloc[:n]
    rows.append({
        "called_share": share,
        "prospects_called": n,
        "subscribers_captured": int(selected["actual_yes"].sum()),
        "capture_rate_of_all_subscribers": selected["actual_yes"].sum() / ranked["actual_yes"].sum(),
        "conversion_rate_within_called_group": selected["actual_yes"].mean(),
    })

targeting_table = pd.DataFrame(rows).round(3)
display(targeting_table)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    targeting_table["called_share"],
    targeting_table["capture_rate_of_all_subscribers"],
    marker="o",
    label="Subscriber capture rate",
)
plt.plot(
    targeting_table["called_share"],
    targeting_table["conversion_rate_within_called_group"],
    marker="o",
    label="Conversion rate among called prospects",
)
plt.xlabel("Share of prospects called")
plt.ylabel("Rate")
plt.title("Capacity-based targeting performance on holdout sample")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Train final deployment model

After evaluation, the final Random Forest can be refit on all available labelled records for deployment. This final refit should not be used to claim a new test performance number because it has now seen all available historical data.

In [ ]:
final_model = models["Random Forest"]
final_model.fit(X, y)

MODEL_PATH = "term_deposit_campaign_model.pkl"
joblib.dump(final_model, MODEL_PATH)
print(f"Saved final model to {MODEL_PATH}")

## 11. Prediction template for new campaign data

The function below scores new prospects and returns predicted class plus subscription probability. In production, prospects should be ranked by `predicted_probability_yes`.

In [ ]:
def score_new_prospects(new_data: pd.DataFrame, model=final_model) -> pd.DataFrame:
    """Return class predictions and subscription probabilities for new prospects."""
    scored = new_data.copy()
    if "duration" in scored.columns:
        scored = scored.drop(columns=["duration"])
    if "y" in scored.columns:
        scored = scored.drop(columns=["y"])

    predicted_class = model.predict(scored)
    predicted_probability_yes = model.predict_proba(scored)[:, list(model.classes_).index("yes")]

    output = new_data.copy()
    output["predicted_class"] = predicted_class
    output["predicted_probability_yes"] = predicted_probability_yes
    return output.sort_values("predicted_probability_yes", ascending=False)

# Example only: score a small sample from the holdout feature matrix.
example_scored = score_new_prospects(X_holdout.head(5))
display(example_scored[["predicted_class", "predicted_probability_yes"]])

## 12. Portfolio conclusion

This project demonstrates a complete business analytics workflow:

1. framing a commercial problem around scarce campaign capacity;
2. controlling leakage by excluding post-call duration from predictive modelling;
3. comparing models against a majority-class benchmark;
4. selecting a model using metrics appropriate for class imbalance;
5. translating probability scores into a ranked call-list strategy; and
6. highlighting monitoring and governance requirements before deployment.

The key business recommendation is to use model probabilities to prioritise calls, not to treat the model as an automatic decision system.